# LAB 4c: Keras Wide & Deep モデルの作成

**学習目標**

1. CSV の列名、ラベル列、列のデフォルト値を設定する
1. CSV ファイルから特徴量とラベルのデータセットを作成する
1. 生の特徴量に対する入力層を作成する
1. 生の特徴量を変換するための前処理層を定義する
1. Wide 層、Deep Dense 隠れ層、および出力層を作成する
1. カスタム評価指標を作成する
1. すべてのパーツを結合して Wide & Deep モデルを構築する
1. トレーニングと評価を行う


## はじめに
このノートブックでは、Keras を使用して Wide & Deep モデルを作成し、赤ちゃんが生まれる前の体重を予測します。

まず、データ入力用の CSV 列名、ラベル列、および列のデフォルト値を定義します。次に、CSV ファイルから特徴量とラベルを含む `tf.data Dataset` を構築し、生の特徴量用の入力層を作成します。続いて、モデル入力用の前処理層を設定し、Keras で Wide & Deep ニューラルネットワークを構築します。カスタム評価指標を作成し、Wide & Deep モデルを構築します。最後に、モデルのトレーニングと評価を行います。

各学習目標はこの受講者用ラボノートブックの __#TODO__ に対応しています。まずはこのノートブックを自分で完成させ、その後に[解答ノートブック](../solutions/4c_keras_wide_and_deep_babyweight.ipynb)を確認してください。

## 必要なライブラリの読み込み

In [ ]:
import datetime
import os
import shutil

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

## CSV ファイルが存在することの確認

このシリーズの 7 番目のラボ [4a_sample_babyweight](../solutions/4a_sample_babyweight.ipynb) では、BigQuery から train、eval、test の CSV ファイルをサンプリングしました。それらのファイルが存在することを確認してください。存在しない場合は、そのラボに戻って作成してください。

In [ ]:
%%bash
ls *.csv

In [ ]:
%%bash
head -5 *.csv

## Keras モデルの作成

### ラボタスク #1: CSV の列名、ラベル列、列のデフォルト値の設定

CSV ファイルが存在することを確認できたので、入力関数で使用するいくつかの設定を行います。
* `CSV_COLUMNS`: 列のヘッダー名です。CSV ファイル内の順序と一致していることを確認してください。
* `LABEL_COLUMN`: ラベルとなる列のヘッダー名です。特徴量辞書から取り出す（pop する）ために必要となります。
* `DEFAULTS`: `CSV_COLUMNS` と同じ長さのリストです（つまり、CSV の各列に対するデフォルト値）。各要素は、その CSV 列のデフォルト値を含むリストです。

In [ ]:
# CSV、ラベル、キー列を決定
# TODO: 文字列の列ヘッダーのリストを作成し、順序が一致していることを確認する。
CSV_COLUMNS = [""]

# TODO: ラベル列の文字列名を追加する
LABEL_COLUMN = ""

NUMERICAL_COLUMNS = ["mother_age", "gestation_weeks"]
CATEGORICAL_COLUMNS = ["is_male", "plurality"]

# 各 CSV 列のデフォルト値を設定。
# is_male と plurality は文字列として扱う。
# TODO: デフォルト値を定義する
DEFAULTS = []

### ラボタスク #2: CSV ファイルからの特徴量とラベルのデータセット作成

### CSV ファイルからの特徴量とラベルのデータセット作成

次に、データを読み込むための `input_fn` を記述します。これにより CSV データセットオブジェクトが作成されます。ただし、列を特徴量とラベルに分割する必要があります。これは、データセットに map メソッドを適用し、特徴量テンソルの辞書からラベル列を取り出す（pop する）ことで行えます。

In [ ]:
def features_and_labels(row_data):
    """特徴量辞書から特徴量とラベルを分割します。

    Args:
        row_data: CSV 列名とテンソル値の辞書。
    Returns:
        特徴量テンソルの辞書とラベルテンソル。
    """
    label = row_data.pop(LABEL_COLUMN)

    return row_data, label  # features, label


def decode_csv(line):
    fields = tf.io.decode_csv(line, record_defaults=DEFAULTS)
    return dict(zip(CSV_COLUMNS, fields))


def load_dataset(pattern, batch_size=1, mode="eval"):
    """tf.data API を使用して CSV ファイルからデータセットを読み込みます。

    Args:
        pattern: str, ファイルのリストにグロブ展開するファイルパターン。
        batch_size: int, バッチあたりのサンプル数。
        mode: 訓練（training）か評価（evaluating）かを判定。
    Returns:
        `Dataset` オブジェクト。
    """
    # TextLineDataset を使用してこれらのファイルの内容を読み込む
    # また、interleave を使用して複数のファイルから並列に読み込めるようにする
    dataset = None

    # 各 CSV ファイルを読み込むために decode_csv を呼び出す
    dataset = None
    # 各 CSV ファイルをマッピングするために features_and_labels を呼び出す
    dataset = None
    dataset = dataset.batch(batch_size)
    # 訓練用にシャッフル
    if mode == "train":
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.repeat().prefetch(buffer_size=tf.data.AUTOTUNE)

    return dataset

### ラボタスク #3: 生の特徴量に対する入力層の作成

入力関数によって読み込まれたデータをモデル関数に渡す必要がありますが、どのように連携させればよいでしょうか？以下を定義することで、Keras の入力層 `keras.layers.Input` を使用できます:
* shape: バッチサイズを含まない形状タプル（整数）。例えば、shape=(32,) は期待される入力が 32 次元のベクトルのバッチであることを示します。このタプルの要素は None にすることもでき、'None' 要素は形状が不明な次元を表します。
* name: レイヤーの任意の名前文字列。モデル内で一意である必要があります（同じ名前を 2 回再利用しないでください）。指定しない場合は自動生成されます。
* dtype: 入力で期待されるデータ型を表す文字列（float32、float64、int32 など）。

In [ ]:
def create_input_layers():
    """各特徴量に対する入力層の辞書を作成します。

    Returns:
        各特徴量の `Keras.layers.Input` 層の辞書。
    """
    # TODO: 各 Dense 特徴量に対する keras.layers.Input の辞書を作成する
    deep_inputs = {}

    # TODO: 各 Sparse 特徴量に対する keras.layers.Input の辞書を作成する
    wide_inputs = {}

    inputs = {**wide_inputs, **deep_inputs}

    return inputs

### ラボタスク #4: 入力用の前処理層の定義

次に、前処理層を定義します。`mother_age` と `gestation_weeks` は数値（numeric）である必要があります。その他の `is_male` と `plurality` はカテゴリカル（categorical）である必要があります。

In [ ]:
def transform(inputs, nembeds):
    """変換された入力の辞書を作成します。

    Returns:
        変換された Tensor の辞書
    """

    deep = {}
    wide = {}

    buckets = {
        "mother_age": np.arange(15, 45, 1).tolist(),
        "gestation_weeks": np.arange(17, 47, 1).tolist(),
    }
    bucketized = {}

    for nc in NUMERICAL_COLUMNS:
        deep[nc] = inputs[nc]
        # TODO: NUMERIC_COLUMNS をバケット化し One-hot エンコードする

    # TODO: バケット化された `mother_age` と `gestation_weeks` の交差特徴量を定義する
    crossed = keras.layers.HashedCrossing()()

    # TODO: 交差特徴量の埋め込み。埋め込み次元には `nembeds` 変数を使用する
    deep["age_gestation_embeds"] = keras.layers.Flatten()(
        keras.layers.Embedding()(crossed)
    )

    vocab = {
        "is_male": ["True", "False", "Unknown"],
        "plurality": [
            "Single(1)",
            "Twins(2)",
            "Triplets(3)",
            "Quadruplets(4)",
            "Quintuplets(5)",
            "Multiple(2+)",
        ],
    }

    for cc in CATEGORICAL_COLUMNS:
        # TODO: CATEGORICAL_COLUMNS を One-hot エンコードする
        wide[cc] = keras.layers.StringLookup()(inputs[cc])

    return wide, deep

### ラボタスク #5: Wide & Deep モデルと出力層の作成

入力を機械学習用に準備する方法がわかったので、次はそれらを期待する出力と接続する必要があります。モデルアーキテクチャがこれら 2 つを結びつけます。ここで Wide & Deep モデルを作成します。Wide 側は線形回帰または Dense 層になります。Deep 側には、いくつかの隠れ Dense 層を作成します。これらすべては単一の Dense 出力層で終わります。これは回帰問題なので、出力層の活性化関数と形状が正しいことを確認してください。

In [ ]:
def get_model_outputs(wide_inputs, deep_inputs, dnn_hidden_units):
    """モデルアーキテクチャを作成し、出力を返します。

    Args:
        wide_inputs: モデルの Wide 側への入力として使用される Dense テンソル。
        deep_inputs: モデルの Deep 側への入力として使用される Dense テンソル。
        dnn_hidden_units: 長さが隠れ層の数であり、i 番目の要素が i 番目の層の
            ニューロン数である整数のリスト。
    Returns:
        モデルからの Dense テンソル出力。
    """
    # Deep 側の隠れ層
    layers = [int(x) for x in dnn_hidden_units]
    deep = deep_inputs

    # TODO: Deep 側の DNN モデルを作成する
    deep_out = None

    # TODO: Wide 側の線形モデルを作成する
    wide_out = None

    # 2 つの側を結合
    both = keras.layers.Concatenate(name="both")([deep_out, wide_out])

    # TODO: 最終出力層を作成する
    output = None

    return output

### ラボタスク #6: すべてのパーツを結合して Wide & Deep モデルを構築

素晴らしい！すべてのパーツが揃ったので、あとはそれらを 1 つの Keras Model に結合するだけです。これは分岐やサイド入力のない単純な順伝播モデルではないため、Keras の Sequential Model API は使用できません。代わりに Keras の Functional Model API を使用します。ここでは `keras.models.Model` を使用して入力と出力を指定してモデルを構築し、オプティマイザ、損失関数、評価指標を指定してモデルをコンパイルします。

In [ ]:
def build_wide_deep_model(dnn_hidden_units=[64, 32], nembeds=3):
    """Keras Functional API を使用して Wide & Deep モデルを構築します。

    Returns:
        `keras.models.Model` オブジェクト。
    """
    # 入力層を作成
    inputs = create_input_layers()

    # Wide と Deep の両方のために生の特徴量を変換
    wide, deep = transform(inputs, nembeds)

    # Keras の Functional API では次が必要です: LayerConstructor()(inputs)
    # TODO: Wide 入力と Deep 入力を結合する
    wide_inputs = keras.layers.Concatenate()()
    deep_inputs = keras.layers.Concatenate()()

    # 入力を指定してモデルの出力を取得
    output = get_model_outputs(wide_inputs, deep_inputs, dnn_hidden_units)

    # モデルを構築し、すべてをまとめてコンパイル
    model = keras.models.Model(inputs=inputs, outputs=output)
    rmse = keras.metrics.RootMeanSquaredError(
        name="root_mean_squared_error", dtype=None
    )
    model.compile(optimizer="adam", loss="mse", metrics=[rmse])

    return model


print("Here is our wide and deep architecture so far:\n")
model = build_wide_deep_model()
print(model.summary())

Keras の plot_model ユーティリティを使用して、Wide & Deep ネットワークを可視化できます。

In [ ]:
keras.utils.plot_model(model=model, show_layer_names=True, rankdir="LR")

## モデルの実行と評価

### ラボタスク #7: トレーニングと評価

CSV ファイルからの入力と設計したアーキテクチャを使用して Keras モデルを構築しました。それでは、モデルパラメータをトレーニングし、トレーニングが進むにつれて未知のデータに対してどれだけ良好に機能しているかを追跡するために定期的に評価を実行して、モデルを実行しましょう。train データセットと eval データセットの両方を読み込み、fit メソッドを通じてそれらをモデルに渡す必要があります。データを読み込む際は、適切なパターン、バッチサイズ、モードを指定してください。また、TensorBoard へのコールバックを追加することも忘れないでください。

In [ ]:
TRAIN_BATCH_SIZE = 32
NUM_TRAIN_EXAMPLES = 10000 * 5  # 訓練データセットはリピートされ、先頭に戻ります
NUM_EVALS = 5  # 評価を行う回数
# 適切なサンプルを得るのに十分で、かつ処理が遅くならない程度の量
NUM_EVAL_EXAMPLES = 10000

trainds = load_dataset(
    pattern="train*",
    batch_size=TRAIN_BATCH_SIZE,
    mode="train",
)

evalds = load_dataset(pattern="eval*", batch_size=1000, mode="eval").take(
    count=NUM_EVAL_EXAMPLES // 1000
)

steps_per_epoch = NUM_TRAIN_EXAMPLES // (TRAIN_BATCH_SIZE * NUM_EVALS)

logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=logdir, histogram_freq=1
)

# TODO: モデルのトレーニングを開始するために model.fit 関数を更新する
history = model.fit()

### 損失曲線の可視化

In [ ]:
# プロット
nrows = 1
ncols = 2
fig = plt.figure(figsize=(10, 5))

for idx, key in enumerate(["loss", "root_mean_squared_error"]):
    ax = fig.add_subplot(nrows, ncols, idx + 1)
    plt.plot(history.history[key])
    plt.plot(history.history[f"val_{key}"])
    plt.title(f"model {key}")
    plt.ylabel(key)
    plt.xlabel("epoch")
    plt.legend(["train", "validation"], loc="upper left");

### モデルの保存

In [ ]:
OUTPUT_DIR = "babyweight_trained_wd"
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

EXPORT_PATH = os.path.join(OUTPUT_DIR, TIMESTAMP)

SAVEDMODEL_PATH = os.path.join(EXPORT_PATH, "savedmodel")
KERAS_PATH = os.path.join(EXPORT_PATH, "model.keras")

model.export(EXPORT_PATH)
model.save(KERAS_PATH)

In [ ]:
!ls $EXPORT_PATH

## ラボのまとめ: 
このラボでは、まずデータ入力用の CSV 列名、ラベル列、および列のデフォルト値を定義しました。次に、CSV ファイルから特徴量とラベルを含む tf.data Dataset を構築し、生の特徴量用の入力層を作成しました。続いて、モデル入力用の前処理層を設定し、Keras で Wide & Deep ニューラルネットワークを構築しました。カスタム評価指標を作成し、Wide & Deep モデルを構築しました。最後に、モデルのトレーニングと評価を行いました。

Copyright 2026 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.